# Bodies of Flora - Colab Pipeline

This notebook runs the complete NLP → 2D → 3D botanical generation pipeline.

## Prerequisites

- **GPU Runtime**: Go to Runtime → Change runtime type → GPU (A100 recommended)
- **API Keys**: You'll need a Groq API key (required) and HuggingFace token (optional)

## Steps

1. Run Cell 1 to install dependencies and clone Hunyuan3D-2
2. **RESTART RUNTIME** after Cell 1 completes
3. Run Cell 2 to launch the Gradio UI

## Cell 1: Environment Setup

This cell:
- Installs all required dependencies (PyTorch, Diffusers, Transformers, etc.)
- Forces compatible versions to avoid runtime conflicts
- Clones and installs Tencent Hunyuan3D-2
- Builds custom rasterizer for texture generation

**Important:** RESTART RUNTIME after this cell completes.

In [ ]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

print("=" * 50)
print(" Setting up pipeline")
print("=" * 50)

pip("--force-reinstall", "numpy==1.26.4")
pip("scipy==1.12.0")
pip("torch>=2.1.0", "torchvision>=0.16.0",
    "transformers>=4.44.0", "diffusers>=0.30.0",
    "accelerate>=0.33.0", "safetensors>=0.4.0",
    "sentencepiece", "protobuf", "huggingface_hub>=0.25.0")
pip("Pillow>=10.0.0", "rembg[gpu]>=2.0.50", "opencv-python-headless>=4.8.0")
pip("trimesh>=4.0.0", "pygltflib>=1.16.0")
pip("requests", "groq>=0.4.0", "gradio>=4.0.0", "gradio_client>=1.0.0")

# Clone Hunyuan3D-2
from pathlib import Path
import subprocess

h3d = Path("/content/Hunyuan3D-2")
if not h3d.exists():
    print("\n>>> Cloning Hunyuan3D-2...")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git",
        str(h3d)], check=True)
    req = h3d / "requirements.txt"
    if req.exists():
        pip("-r", str(req))
    print("  ✓ Hunyuan3D-2 cloned")
else:
    print("  ✓ Hunyuan3D-2 already exists")

# Build custom_rasterizer if needed
cr_path = h3d / "hy3dgen" / "texgen" / "custom_rasterizer"
if cr_path.exists():
    print(">>> Building custom_rasterizer...")
    subprocess.run([sys.executable, "setup.py", "install"],
                   cwd=str(cr_path), check=False)

Path("/content/outputs").mkdir(exist_ok=True)

try:
    from google.colab import userdata
    import os
    for k in ["HF_TOKEN", "GROQ_API_KEY"]:
        try:
            v = userdata.get(k)
            if v and not os.environ.get(k): os.environ[k] = v
        except: pass
except: pass

print("\n" + "=" * 50)
print("✅ DONE — RESTART RUNTIME → Run Cell 2")
print("=" * 50)

## Cell 2: Pipeline + Gradio UI

This cell:
- Loads the full multi-stage pipeline
- Launches an interactive Gradio UI

**Pipeline Stages:**
1. **Stage 1a**: Species identification (Groq LLM with comparative reasoning)
2. **Stage 1b**: Botanical enrichment + FLUX prompt generation
3. **Stage 2a**: 2D image generation (FLUX.1-schnell)
4. **Stage 2b**: Image cleaning (background removal, morphological cleanup)
5. **Stage 3**: 3D mesh generation (Hunyuan3D-2)

In [ ]:
import os, re, gc, io, json, sys, time, traceback
from pathlib import Path
from typing import Optional, Dict, Any
import numpy as np
from PIL import Image
import torch
import gradio as gr

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from google.colab import userdata
    for key in ["HF_TOKEN", "GROQ_API_KEY"]:
        try:
            v = userdata.get(key)
            if v and not os.environ.get(key):
                os.environ[key] = v
        except:
            pass
except:
    pass

H3D = Path("/content/Hunyuan3D-2")
if str(H3D) not in sys.path:
    sys.path.insert(0, str(H3D))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "llama-3.1-8b-instant",
    "gemma2-9b-it",
]
DEFAULT_FLUX = "black-forest-labs/FLUX.1-schnell"
OUTDIR = Path("/content/outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)
_flux_pipe = None

print(f">>> Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# Indigenous / Comparative Alias Map
INDIGENOUS_ALIAS_MAP = {
    "miskominagaawanzh": "Sarracenia purpurea",
    "little cranberry-like plant with a vessel": "Sarracenia purpurea",
    "bloodroot": "Sanguinaria canadensis",
    "the root that bleeds": "Sanguinaria canadensis",
    "root that bleeds": "Sanguinaria canadensis",
    "blood root": "Sanguinaria canadensis",
    "pitcher plant": "Sarracenia purpurea",
    "plant that catches flies": "Drosera rotundifolia",
    "fly trap": "Dionaea muscipula",
    "venus flytrap": "Dionaea muscipula",
    "thunder plant": "Podophyllum peltatum",
    "may apple": "Podophyllum peltatum",
    "snake root": "Aristolochia serpentaria",
    "indian pipe": "Monotropa uniflora",
    "ghost plant": "Monotropa uniflora",
}


def normalize_indigenous_input(text: str) -> str:
    raw = text.strip().lower()
    for pattern, species in INDIGENOUS_ALIAS_MAP.items():
        if pattern in raw:
            return f"{species} — originally described as: {text.strip()}"
    return text


def _call_groq(messages, groq_key, temperature=0.2, max_tokens=1400):
    from groq import Groq
    client = Groq(api_key=groq_key)
    last_err = None
    for model in GROQ_MODELS:
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages,
                temperature=temperature, max_tokens=max_tokens)
            txt = resp.choices[0].message.content
            if txt:
                return txt.strip(), model
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Groq failed across all models: {last_err}")


def _extract_json(text):
    text = text.strip()
    try:
        return json.loads(text)
    except:
        pass
    c = re.sub(r'```(?:json)?\s*', '', text)
    c = re.sub(r'```\s*$', '', c, flags=re.MULTILINE)
    try:
        return json.loads(c.strip())
    except:
        pass
    m = re.search(r"\{.*\}", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except:
            pass
    raise ValueError("No valid JSON found in model response")


STAGE1A_SYSTEM = """You are an expert ethnobotanist, taxonomic botanist, and historical flora interpreter.

CRITICAL — READ THIS FIRST:
Many plant descriptions use COMPARATIVE language where each word is a CLUE,
not a literal species reference. You MUST reason through these clues.

COMPARATIVE REASONING PROTOCOL (do this BEFORE identifying):
When input contains words like "-like", "similar to", "resembling", or descriptive
features ("with a vessel", "with sticky leaves", "with umbrella leaves"):

1. DECOMPOSE each phrase:
   - Size/habit comparisons ("-like", "small", "little") = ecological niche clue
   - Structural features ("vessel", "pitcher", "trap", "pouch", "cup") = morphology clue
   - Color references ("bleeds red", "white sap") = chemical/visual clue
   - Habitat hints ("bog", "marsh", "swamp") = ecological clue

2. CROSS-REFERENCE the clues:
   - What plant has ALL these features simultaneously?
   - The comparison word is usually about SIZE/HABITAT, not the species itself

KEY VOCABULARY from indigenous/descriptive names:
- "vessel", "pitcher", "cup", "jug" → pitcher plant (Sarracenia, Nepenthes, Darlingtonia)
- "trap", "catch", "eat insects", "sticky" → carnivorous plant
- "bleeding", "blood", "red sap" → Sanguinaria, Chelidonium
- "umbrella leaves" → Podophyllum, Diphylleia
- "ghost", "corpse", "no leaves" → mycoheterotrophic (Monotropa)

RULES:
- Show your reasoning chain in the "reasoning" field
- Prefer specific identification over generic guesses
- NEVER leave species_name empty

Return ONLY valid JSON:
{
  "species_name": "Scientific name — NEVER empty",
  "common_name": "common name",
  "family": "botanical family",
  "confidence": 0.0-1.0,
  "reasoning": ["step 1...", "step 2...", "step 3..."],
  "morphology": {
    "growth_form": "herb/shrub/tree/vine/rosette/etc",
    "flower": "description or empty",
    "leaf": "description or empty",
    "fruit_seed": "description or empty",
    "root": "description or empty",
    "stem": "description or empty",
    "special_structures": "pitchers/traps/tendrils/etc or empty",
    "habitat": "bog/forest/prairie/etc or empty"
  },
  "part_focus": "whole plant | flower | seed | root | leaf | fruit | unknown"
}"""


def stage1a_nlp(user_text: str, groq_key: str) -> dict:
    t0 = time.time()
    print("━━━ Stage 1a: Species identification ━━━")
    txt, model = _call_groq(
        [{"role": "system", "content": STAGE1A_SYSTEM},
         {"role": "user", "content": f"Identify this plant:\n\n{user_text}"}],
        groq_key=groq_key, temperature=0.1, max_tokens=1400)
    data = _extract_json(txt)
    data.setdefault("species_name", "Unknown plant")
    data.setdefault("common_name", "")
    data.setdefault("family", "")
    data.setdefault("confidence", 0.5)
    data.setdefault("reasoning", [])
    data.setdefault("morphology", {})
    data.setdefault("part_focus", "whole plant")
    dt = time.time() - t0
    print(f"  Model: {model} ({dt:.1f}s)")
    print(f"  Species: {data['species_name']}")
    print(f"  Common: {data['common_name']}")
    print(f"  Confidence: {data['confidence']}")
    return data


STAGE1B_SYSTEM_TEMPLATE = """You are a botanical image prompt engineer.

The plant is identified as:
- Species: {species}
- Common name: {common}
- Family: {family}
- Part focus: {part_focus}

Morphology from Stage 1a:
{morph_text}

YOUR JOB: Build a FLUX image generation prompt that is:
1. Visually faithful to this SPECIFIC species
2. Scientifically accurate
3. Clear for 3D reconstruction — three-quarter angle, isolated specimen

RULES:
- DO NOT change the identified species
- One isolated specimen on clean white studio background
- 80-150 words in the flux_prompt

Return ONLY valid JSON:
{{
  "species_name": "{species}",
  "visual_summary": "2-sentence description",
  "clip_hint": "12-word visual summary for CLIP encoder",
  "flux_prompt": "Three-quarter angle photorealistic 3D render of ... [80-150 words]",
  "negative_prompt": "things to exclude from the image"
}}"""


def stage1b_enrich(stage1a_output: dict, groq_key: str) -> dict:
    t0 = time.time()
    print("━━━ Stage 1b: Enrichment + FLUX prompt ━━━")
    species = stage1a_output.get("species_name", "Unknown plant")
    common = stage1a_output.get("common_name", "")
    family = stage1a_output.get("family", "")
    part_focus = stage1a_output.get("part_focus", "whole plant")
    morph = stage1a_output.get("morphology", {})
    morph_lines = []
    for key in ["growth_form", "flower", "leaf", "fruit_seed", "root", "stem", "special_structures", "habitat"]:
        val = morph.get(key, "")
        if val and isinstance(val, str) and val.strip():
            morph_lines.append(f"  {key}: {val}")
    morph_text = "\n".join(morph_lines) if morph_lines else "  (minimal)"
    system_prompt = STAGE1B_SYSTEM_TEMPLATE.format(
        species=species, common=common, family=family,
        part_focus=part_focus, morph_text=morph_text)
    user_prompt = f"Build the FLUX prompt for {species}" + (f" ({common})" if common else "") + f", focus: {part_focus}"
    try:
        txt, model = _call_groq(
            [{"role": "system", "content": system_prompt},
             {"role": "user", "content": user_prompt}],
            groq_key=groq_key, temperature=0.15, max_tokens=1200)
        data = _extract_json(txt)
        dt = time.time() - t0
        print(f"  Model: {model} ({dt:.1f}s)")
    except Exception as e:
        print(f"  ⚠ Enrichment failed: {e}")
        data = {}
    data.setdefault("species_name", species)
    data.setdefault("clip_hint", f"3D render {common or species} botanical specimen")
    if not data.get("negative_prompt"):
        data["negative_prompt"] = "multiple plants, bouquet, pot, vase, soil, landscape, text, watermark, blurry"
    if not data.get("flux_prompt") or len(data["flux_prompt"].split()) < 15:
        morph_desc = ", ".join(f"{k}: {v}" for k, v in morph.items() if v and isinstance(v, str) and v.strip())
        data["flux_prompt"] = (
            f"Three-quarter angle photorealistic 3D render of {species}" +
            (f" ({common})" if common else "") + f". {morph_desc}. " +
            f"Single isolated botanical specimen, {part_focus}, clean white studio background, volumetric lighting, sharp botanical detail."
        )
    print(f"  Prompt: {data['flux_prompt'][:150]}...")
    return data


def _load_flux(hf_token=""):
    global _flux_pipe
    if _flux_pipe is not None:
        return _flux_pipe
    print("  Loading FLUX.1-schnell (CPU offload)...")
    if hf_token and hf_token.strip():
        os.environ["HF_TOKEN"] = hf_token.strip()
    from diffusers import FluxPipeline
    token = os.environ.get("HF_TOKEN", None)
    _flux_pipe = FluxPipeline.from_pretrained(DEFAULT_FLUX, torch_dtype=torch.float16, token=token)
    _flux_pipe.enable_model_cpu_offload()
    print("  ✓ FLUX ready (CPU offload)")
    return _flux_pipe


def _unload_flux():
    global _flux_pipe
    if _flux_pipe is not None:
        del _flux_pipe
        _flux_pipe = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def stage2a_flux(prompt: str, clip_hint: str = "", seed: int = 42, hf_token: str = "") -> str:
    t0 = time.time()
    print("━━━ Stage 2a: FLUX.1-schnell ━━━")
    pipe = _load_flux(hf_token=hf_token)
    gen = torch.Generator("cpu").manual_seed(int(seed))
    try:
        with torch.inference_mode():
            out = pipe(prompt=clip_hint if clip_hint else prompt, prompt_2=prompt,
                       guidance_scale=0.0, num_inference_steps=4, max_sequence_length=256,
                       generator=gen, height=1024, width=1024)
    except RuntimeError:
        gc.collect()
        torch.cuda.empty_cache()
        print("  OOM → retrying 768×768")
        gen = torch.Generator("cpu").manual_seed(int(seed))
        with torch.inference_mode():
            out = pipe(prompt=clip_hint if clip_hint else prompt, prompt_2=prompt,
                       guidance_scale=0.0, num_inference_steps=4, max_sequence_length=256,
                       generator=gen, height=768, width=768)
    img = out.images[0].convert("RGB")
    p = OUTDIR / f"flux_{int(time.time())}_{seed}.png"
    img.save(p, "PNG")
    print(f"  ✓ Generated ({time.time() - t0:.1f}s)")
    return str(p)


def stage2b_clean(image_path: str) -> str:
    t0 = time.time()
    print("━━━ Stage 2b: Image cleaning ━━━")
    from rembg import remove
    import cv2
    img = Image.open(image_path).convert("RGB")
    arr = np.array(img)
    res = remove(arr)
    if isinstance(res, bytes):
        rgba = np.array(Image.open(io.BytesIO(res)).convert("RGBA"))
    else:
        rgba = np.array(Image.fromarray(res).convert("RGBA"))
    rgb = rgba[:, :, :3].astype(np.float32)
    bright = rgb.mean(axis=2)
    var = rgb.var(axis=2)
    rgba[(bright > 230) & (var < 20), 3] = 0
    rgba[(bright < 15) & (var < 10), 3] = 0
    alpha = rgba[:, :, 3]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    alpha = cv2.morphologyEx(alpha, cv2.MORPH_CLOSE, k, iterations=2)
    alpha = cv2.morphologyEx(alpha, cv2.MORPH_OPEN, k, iterations=1)
    alpha[alpha < 128] = 0
    alpha[alpha >= 128] = 255
    rgba[:, :, 3] = alpha
    pil = Image.fromarray(rgba)
    bbox = pil.getbbox()
    SZ = 1024
    if bbox:
        cr = pil.crop(bbox)
        cr.thumbnail((int(SZ * 0.85), int(SZ * 0.85)), Image.LANCZOS)
        canvas = Image.new("RGBA", (SZ, SZ), (255, 255, 255, 255))
        canvas.paste(cr, ((SZ - cr.width) // 2, (SZ - cr.height) // 2), cr)
        out = Image.new("RGB", (SZ, SZ), (255, 255, 255))
        out.paste(canvas, mask=canvas.split()[-1])
    else:
        out = img.resize((SZ, SZ))
    p = OUTDIR / f"clean_{int(time.time())}.png"
    out.save(p, "PNG")
    print(f"  ✓ Cleaned ({time.time() - t0:.1f}s)")
    return str(p)


def stage3_hunyuan(image_path: str, seed: int = 42, label: str = "plant") -> str:
    t0 = time.time()
    print("━━━ Stage 3: Hunyuan3D-2 local ━━━")
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
    import trimesh as tm
    pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
        "tencent/Hunyuan3D-2", subfolder="hunyuan3d-dit-v2-0",
        torch_dtype=torch.float16, device=DEVICE)
    try:
        pipe.enable_flashvdm(mc_algo='mc')
        print("  FlashVDM enabled")
    except Exception as e:
        print(f"  FlashVDM not available: {e}")
    image = Image.open(image_path).convert("RGB")
    gen = torch.Generator(device=DEVICE).manual_seed(int(seed))
    with torch.no_grad():
        mesh = pipe(image=image, num_inference_steps=50, guidance_scale=7.5,
                    octree_resolution=384, num_chunks=200000, generator=gen, output_type="trimesh")
    if isinstance(mesh, list):
        meshes = [m for m in mesh if isinstance(m, tm.Trimesh)]
        if not meshes:
            raise ValueError("Hunyuan3D returned no trimesh objects")
        mesh = meshes[0] if len(meshes) == 1 else tm.util.concatenate(meshes)
    try:
        if hasattr(mesh, 'remove_degenerate_faces'):
            mesh.remove_degenerate_faces()
        if hasattr(mesh, 'remove_unreferenced_vertices'):
            mesh.remove_unreferenced_vertices()
        tm.repair.fix_normals(mesh)
        if hasattr(mesh, 'split'):
            comps = mesh.split()
            if len(comps) > 1:
                total = sum(len(c.faces) for c in comps)
                kept = [c for c in comps
                        if sorted(c.bounding_box.extents)[0] / max(sorted(c.bounding_box.extents)[2], 1e-8) >= 0.08
                        or len(c.faces) / total > 0.5]
                if kept:
                    mesh = tm.util.concatenate(kept)
        center = (mesh.bounds[0] + mesh.bounds[1]) / 2
        mesh.vertices -= center
        scale = np.max(mesh.bounds[1] - mesh.bounds[0])
        if scale > 1e-8:
            mesh.vertices /= scale
        tm.smoothing.filter_laplacian(mesh, iterations=1)
    except Exception as e:
        print(f"  ⚠ Mesh cleanup partial: {e}")
    safe_label = re.sub(r"[^a-zA-Z0-9_-]+", "_", label).strip("_") or "plant"
    glb_path = OUTDIR / f"{safe_label}_{int(time.time())}.glb"
    mesh.export(str(glb_path))
    print(f"  ✓ Mesh saved ({time.time() - t0:.1f}s): {glb_path}")
    return str(glb_path)


def run(user_text: str, groq_key: str, hf_token: str = "", seed: int = 42):
    logs = []
    def log(msg):
        print(msg)
        logs.append(str(msg))
    try:
        if not user_text or not user_text.strip():
            raise ValueError("Empty plant input")
        if not groq_key or not groq_key.strip():
            raise ValueError("Missing GROQ_API_KEY")
        os.environ["GROQ_API_KEY"] = groq_key.strip()
        if hf_token and hf_token.strip():
            os.environ["HF_TOKEN"] = hf_token.strip()
        seed = int(seed)
        normalized_text = normalize_indigenous_input(user_text)
        if normalized_text != user_text:
            log(f"  Indigenous alias matched: {user_text.strip()}")
            log(f"  → Normalized to: {normalized_text}")
        log(f"Input: {user_text[:100]}...")
        log("\n▸ Stage 1a — Species identification")
        stage1a = stage1a_nlp(normalized_text, groq_key.strip())
        log(f"  → {stage1a['species_name']} ({stage1a.get('common_name', '')})")
        log(f"  Confidence: {stage1a.get('confidence', '?')}")
        log("\n▸ Stage 1b — Botanical enrichment + FLUX prompt")
        stage1b = stage1b_enrich(stage1a, groq_key.strip())
        flux_prompt = stage1b.get("flux_prompt", "")
        clip_hint = stage1b.get("clip_hint", "")
        if not flux_prompt:
            raise ValueError("Stage 1b returned no FLUX prompt")
        log(f"  Prompt: {flux_prompt[:150]}...")
        log("\n▸ Stage 2a — FLUX image generation")
        img_path = stage2a_flux(flux_prompt, clip_hint=clip_hint, seed=seed, hf_token=hf_token)
        log("\n▸ Stage 2b — Image cleaning")
        clean_path = stage2b_clean(img_path)
        log("\n▸ Freeing FLUX memory for Hunyuan3D")
        _unload_flux()
        label = stage1a.get("species_name", "plant")
        log("\n▸ Stage 3 — Hunyuan3D-2 local mesh generation")
        glb_path = stage3_hunyuan(clean_path, seed=seed, label=label)
        final_json = {
            "input_text": user_text,
            "normalized_input": normalized_text if normalized_text != user_text else None,
            "stage1a_species_id": stage1a,
            "stage1b_enrichment": {k: v for k, v in stage1b.items() if k != "flux_prompt"},
            "flux_prompt": flux_prompt,
            "outputs": {"image_raw": img_path, "image_clean": clean_path, "mesh_glb": glb_path},
            "seed": seed,
        }
        log("\n✓ Pipeline complete")
        return (json.dumps(final_json, indent=2, ensure_ascii=False), img_path, clean_path, glb_path, "\n".join(logs))
    except Exception as e:
        tb = traceback.format_exc()
        logs.append(f"\n✗ ERROR: {e}\n{tb}")
        return (json.dumps({"error": str(e), "traceback": tb}, indent=2), None, None, None, "\n".join(logs))


def _ui(text, groq_key, hf_token, seed):
    if not text.strip():
        raise gr.Error("Enter a plant description")
    if not groq_key.strip():
        raise gr.Error("Groq API key required")
    nlp_json, img, clean, glb, log_text = run(text, groq_key, hf_token, int(seed))
    gallery = []
    if img and os.path.exists(img):
        gallery.append((img, "FLUX 2D"))
    if clean and os.path.exists(clean):
        gallery.append((clean, "Cleaned"))
    files = [f for f in [img, clean, glb] if f and os.path.exists(f)]
    return nlp_json, gallery, glb, files, log_text


with gr.Blocks(title="Bodies of Flora") as demo:
    gr.Markdown("""# Bodies of Flora\n\nTransform natural language plant descriptions into 3D botanical models.""")
    text_in = gr.Textbox(label="Plant Input", lines=8,
        placeholder="Describe a plant using scientific names, common names, indigenous names, or descriptive features.")
    run_btn = gr.Button("Generate", variant="primary", size="lg")
    with gr.Accordion("Config", open=False):
        groq_key = gr.Textbox(label="Groq API Key", type="password", value=os.environ.get("GROQ_API_KEY", ""))
        hf_token = gr.Textbox(label="HuggingFace Token", type="password", value=os.environ.get("HF_TOKEN", ""))
        seed_sl = gr.Slider(0, 999999, value=42, step=1, label="Seed")
    with gr.Tabs():
        with gr.Tab("NLP Output"):
            nlp_out = gr.Code(label="Species + reasoning + enrichment + prompts", language="json")
        with gr.Tab("2D Images"):
            gallery = gr.Gallery(label="FLUX → Cleaned", columns=2, height=420)
        with gr.Tab("3D Model"):
            model3d = gr.Model3D(label="3D mesh", clear_color=[0.85, 0.85, 0.85, 1.0])
    files_out = gr.Files(label="Downloads")
    log_out = gr.Textbox(label="Pipeline Log", lines=25)
    run_btn.click(fn=_ui, inputs=[text_in, groq_key, hf_token, seed_sl],
                  outputs=[nlp_out, gallery, model3d, files_out, log_out])

demo.queue(max_size=10).launch(share=True, debug=False, show_error=True)